## No-Op Profiler Example in Python

### 1️⃣ What It Does
- Acts like a profiler, but **does nothing**.
- Lets you **keep profiling calls in your code** without adding overhead.
- You can **swap it with a real profiler** later if needed.


In [ ]:
class NoOpProfiler:
    def __enter__(self):
        # Called at the start of a 'with' block
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        # Called at the end of the 'with' block
        pass

    def start(self):
        pass

    def stop(self):
        pass

In [ ]:
def work(n):
    data = [i*i for i in range(n)]
    return sum(data)

# Wrap code in the no-op profiler
with NoOpProfiler() as prof:
    prof.start()
    result = work(1000000)
    prof.stop()

print(result)

## When a No-Op Profiler Makes Sense

### 1️⃣ Why Use a No-Op Profiler?

- It **lets you keep the profiling code in your project** without actually measuring anything.
- Useful in **shared code** or **libraries** where profiling may be turned on later.
- Makes it **easy to swap in a real profiler** without touching functions.
- Keeps **development and production code compatible**.

---

### 2️⃣ When It’s Not Needed

- If you are **just writing a simple script** or notebook and **won’t profile it later**, a no-op profiler adds **no runtime benefit**.
- You can skip it and only add `cProfile`, `line_profiler`, or `memory_profiler` when you **actually need to measure performance**.

---

### 3️⃣ Practical Advice

- **Small scripts / notebooks:** just use the real profiler when needed.
- **Larger projects / libraries:** use a no-op profiler at first, then swap to a real profiler for testing or optimization.

> ✅ **Summary:**
> No-op profiler is **about code structure and flexibility**, not speed. If you don’t need profiling yet, you can safely ignore it.

In [ ]:
## Swapping No-Op Profiler with Real Profiler

### Define a Real Profiler Wrapper

import cProfile
import io
import pstats

class RealProfiler:
    def __enter__(self):
        self.profiler = cProfile.Profile()
        self.profiler.enable()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.profiler.disable()
        s = io.StringIO()
        ps = pstats.Stats(self.profiler, stream=s).sort_stats("tottime")
        ps.print_stats(5)  # print top 5 functions
        print(s.getvalue())

    def start(self):
        self.profiler.enable()

    def stop(self):
        self.profiler.disable()

In [ ]:
def work(n):
    data = [i*i for i in range(n)]
    return sum(data)

# Swap the profiler here
Profiler = RealProfiler  # previously it was NoOpProfiler

with Profiler() as prof:
    prof.start()
    result = work(1000000)
    prof.stop()

print(result)

## Why Adding a Profiler Later Can Be Slightly Harder

Function Wrapping

If you already have code calling functions everywhere:

```python
result = work(1_000_000)
another_result = analyze(data)
```
-----
To profile later, you either need to:

Manually wrap every function with a profiler, e.g.:
```
import cProfile
cProfile.run('work(1000000)')
```
If profiling is added later, you need to remember to add it consistently in multiple places.

Otherwise, some functions are profiled, some are not — results may be misleading.